In [ ]:
import pandas as pd
import numpy as np
import arviz as az
from jax import numpy as jnp
from jax.random import PRNGKey
import numpyro
from numpyro import distributions as dist
from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import Parameter, defer, CompartmentValues, Time
from summer3.epi import CompartmentalModelODE, CategoryData, \
    strat_data_from_pandas, build_istate, dti_to_epoch, TransitionFlow, \
    CategoryGroup, StratSpec, ManagedArray

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, INFECT_COMPS, MAX_AGE, \
    INF_STRATA
from tb_macro.epi import get_base_model, add_natural_history, \
    add_health_system_flows, add_seeding, add_detection
from tb_macro.inputs import get_country_pop, get_single_age_pop_from_ungroups, \
    get_group_popsizes, get_un_mortality, load_conmat, convert_conmat, \
    normalise_spectral_radius, add_groups_to_single_pop, build_age_weight_lookup, \
    get_fertility_data
from tb_macro.mixing import get_norm_c_matrix
from tb_macro.demography import add_replacement_deaths, add_ageing_flows, \
    prepare_pop_data_for_entries, add_entry_births, make_death_func

In [ ]:
iso3 = "VNM"
start_year = 1950

pop_data = get_country_pop(iso3)
single_age = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age)
agg_pop_data = group_popsize.sum(axis=1)
mort_data = get_un_mortality(iso3, start_year)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
conmat_data = load_conmat(iso3)
matrix = convert_conmat(conmat_data)
norm_matrix = normalise_spectral_radius(matrix)
add_groups_to_single_pop(single_age)
age_weights = build_age_weight_lookup(single_age)
fert = get_fertility_data(iso3)
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)

In [ ]:
def infect_process(
    compartment_values: ManagedArray,
    contact_rate: float,
    freq_dens_exponent: float,
    age_cats: CategoryGroup,
    infectious_compartments: StratSpec,
    age_breaks: jnp.array,
    young_end_age: int,
    young_suscept: float,
    mm_function: callable,
    a_spread: float,
    bg_mixing: float,
    pc_strength: float,
    weights: jnp.array,
    weight_ends: jnp.array,
    pops: jnp.array,
    pop_ends: jnp.array,
    fert: jnp.array,
    fert_ends: jnp.array,
    time: float,
):
    time = 2000 # temporary patch for speed

    infector_cats = age_cats
    infectee_cats = age_cats
    infect_pop_cats = infector_cats.product(infectious_compartments)
    age_infect = jnp.where(age_breaks < young_end_age, 0.0, 1.0)
    age_suscept = jnp.where(age_breaks < young_end_age, young_suscept, 1.0)
    ipops = compartment_values.sumcats(infect_pop_cats)
    total_pop = compartment_values.sumcats(infector_cats)
    inf_pressure = contact_rate * age_infect * ipops.data / total_pop.data ** freq_dens_exponent
    mm = mm_function(weights, weight_ends, pops, pop_ends, fert, fert_ends, time, bg_mixing, a_spread, pc_strength)
    age_foi = age_suscept * (mm @ inf_pressure)
    return CategoryData(infectee_cats, age_foi)

In [ ]:
start_time = 1800.0
end_time = 2000.0
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(start_time, end_time)
young_end_age = 15
start_pops = [1000.0] * len(AGE_STRATA)
times, rates = prepare_pop_data_for_entries(agg_pop_data, start_time, sum(start_pops))

for comp in INFECT_COMPS:
    suscept_comp = "cleared" if comp in ["cleared", "recovered"] else comp
    reinfect_foi = defer(infect_process)(
        CompartmentValues, 
        Parameter("contact_rate", 0.0) * Parameter(f"rel_sus_{suscept_comp}", 0.0), 
        Parameter("freq_dens_exponent", 1.0), 
        age_strat.categories(), 
        disease_state["active"],
        jnp.array(AGE_STRATA),
        young_end_age,
        Parameter("young_suscept", 0.0),
        get_norm_c_matrix,
        Parameter("a_spread", 0.0),
        Parameter("bg_mixing", 0.0),
        Parameter("pc_strength", 0.0),
        jnp.array(age_weights),
        jnp.array(age_weights.index[[0, -1]]),
        jnp.array(group_popsize),
        jnp.array(group_popsize.index[[0, -1]]),
        jnp.array(fert_padded),
        jnp.array(fert_padded.index[[0, -1]]),
        Time,
    )
    reinfect = TransitionFlow(
        f"infect_{comp}",
        disease_state[comp],
        disease_state["incipient"],
        reinfect_foi,
    )
    epi_model.add_flow(reinfect)

add_natural_history(epi_model, disease_state, clin_strat, infect_strat)
add_health_system_flows(epi_model, disease_state, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state)
add_detection(epi_model, disease_state, clin_strat)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, start_time)
add_entry_births(epi_model, disease_state, age_strat, start_time, rates, times)

In [ ]:
for age in AGE_STRATA:
    latency_age_cat = "age0" if age < 5 else "age5" if age < 15 else "age15"
    contain = TransitionFlow(
        f"containment_{age}",
        (disease_state["incipient"], age_strat[str(age)]),
        (disease_state["contained"], age_strat[str(age)]),
        Parameter(f"containment_{latency_age_cat}", 0.0),
    )
    epi_model.add_flow(contain)

    for strat in INF_STRATA:
        prop_inf = Parameter("progression_prop_infectious", 0.0)
        strat_prop = prop_inf if strat == "high" else 1.0 - prop_inf
        progression = TransitionFlow(
            f"progression_{strat}_{age}",
            (disease_state["incipient"], age_strat[str(age)]),
            (clin_strat["subclin"], infect_strat[strat], age_strat[str(age)]),
            Parameter(f"progression_{latency_age_cat}", 0.0) * strat_prop,
        )
        epi_model.add_flow(progression)

In [ ]:
def get_treatment_outcomes(death_times, death_data, rx_duration, tsr, prop_neg_rx_death, time, start_time):
    death_func = make_death_func(death_times, death_data, start_time)
    death_rates = death_func(time)
    prop_natural_death_on_rx = 1.0 - jnp.exp(-rx_duration * death_rates)
    req_prop_death_on_rx = (1.0 - tsr) * prop_neg_rx_death
    prop_death_from_rx = jnp.maximum(req_prop_death_on_rx, 0.0)
    return 1.0 - tsr - prop_death_from_rx - prop_natural_death_on_rx

def get_relapse_rate(death_times, death_rates, rx_duration, tsr, prop_neg_rx_death, time, start_time):
    relapse_prop = get_treatment_outcomes(death_times, death_rates, rx_duration, tsr, prop_neg_rx_death, time, start_time)
    return relapse_prop / rx_duration

In [ ]:
for age in AGE_STRATA:
    d_times = death_rates.index.to_numpy(dtype=float)
    d_rates = death_rates[age].to_numpy(dtype=float)
    rx_dur = Parameter("rx_duration", 0.0)
    tsr = Parameter("tsr", 0.0)
    prop_neg_rx_death = Parameter("prop_neg_rx_death", 0.0)
    relapse = TransitionFlow(
        f"relapse_{age}",
        (disease_state["treatment"], age_strat[str(age)]),
        (clin_strat["subclin"], infect_strat["low"], age_strat[str(age)]),
        defer(get_relapse_rate)(d_times, d_rates, rx_dur, tsr, prop_neg_rx_death, Time, start_time),
    )
    epi_model.add_flow(relapse)

In [ ]:
age_strings = [str(a) for a in AGE_STRATA]
data = pd.Series(index=age_strings, data=np.array(start_pops))
base_pops = strat_data_from_pandas(data, age_strat)
init_splits = [0.0] * len(ALL_COMPARTMENTS)
init_splits[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_splits)))]
epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
base_params = {
    "contact_rate": 0.01,
    "freq_dens_exponent": 0.5392614891179714,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 0.3324848208129065, 
    "rel_sus_cleared": 0.7101012998861186, 
    "containment_age0": 4.4,
    "containment_age5": 4.4,
    "containment_age15": 2.0,
    "clearance_rate": 0.05643858771640714,
    "breakdown_rate": 0.5678500778834155,
    "progression_age0": 2.4,
    "progression_age5": 2.0,
    "progression_age15": 0.1,
    "progression_prop_infectious": 0.5,
    "increase_infect": 2.799432998282645,
    "decrease_infect": 1.0,
    "clinical_development": 2.280456422265371,
    "clinical_regression": 1.0,
    "self_recovery": 0.4,
    "treatment_recovery": 0.5,
    "treatment_relapse": 0.0,
    "seed_peak_time": 30.0,
    "seed_peak_rate": 0.01,
    "seed_duration": 10.0,
    "recent_detection_rate": 3.1435769682295085,
    "passive_detection_shape": 0.12804084477668393,
    "passive_detection_inflection": 56.359353581523,
    "passive_detection_past_frac": 0.7117933876877114,
    "a_spread": 9.832801834382463,
    "bg_mixing": 0.029353135750061505,
    "pc_strength": 0.9853729910300592,
    "young_suscept": 0.5,
    "rx_duration": 0.5,
    "tsr": 0.7,
    "prop_neg_rx_death": 0.1,
}
runner, istate = get_runner(epi_model)
results = epi_model.run(base_params)

In [ ]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().plot()

In [ ]:
results["compartments"].sumcats(compartment=age_strat.categories()).to_pandas_df().plot()

In [ ]:
# def get_derived_results(params):
#     results = epi_model.run(params)  # runner.run(istate.data, params)
#     inf_flow = results["flows"]["infect_mtb_naive"]
#     weekly_target = (
#         inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
#     )
#     return weekly_target

In [ ]:
# priors = {
#     "contact_rate": dist.Uniform(0.001, 0.5),
# }

In [ ]:
# def model():
#     params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
#     weekly_modelled = get_derived_results(params)

#     ll = dist.Poisson(inf_target_fuzzy.to_numpy()).log_prob(weekly_modelled.data)
#     numpyro.factor("ll", ll)

In [ ]:
# kernel = infer.NUTS(model)
# mcmc = infer.MCMC(kernel, num_warmup=200, num_samples=200, num_chains=4)
# k = PRNGKey(0)
# mcmc.run(k)

In [ ]:
# idata = az.from_numpyro(mcmc)

In [ ]:
# az.summary(idata)

In [ ]:
# az.plot_posterior(idata)

In [ ]:
# az.plot_trace(idata, compact=False)